# Analisis Daya Beli per Provinsi

## Tujuan
Notebook ini membandingkan beberapa pendekatan econometrics dan machine learning untuk membaca hubungan antara inflasi, pasar kerja, pendapatan, dan daya beli rumah tangga.

## Cakupan aktif
- Unit analisis: provinsi
- Periode deployment: 2021-2025
- Target: `Total_Pengeluaran`
- Fokus akhir: artefak Ridge yang dipakai aplikasi web

## Hasil yang ingin dicapai
- model pembanding yang tetap informatif untuk analisis
- satu jalur retrain deployment yang konsisten dengan website
- ringkasan metrik dan insight kebijakan yang mudah dibaca


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import joblib
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV

from linearmodels.panel import PanelOLS
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Helper function
def print_metrics(y_true, y_pred, label=''):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{label} R2: {r2:.4f} | MAE: Rp {mae:,.0f} | RMSE: Rp {rmse:,.0f}")
    return {'R2': r2, 'MAE': mae, 'RMSE': rmse}

print("Library loaded successfully")


Library loaded successfully


## 1. Load Data dan Orientasi

Bagian ini memuat dataset aktif, memeriksa ukuran data, serta menegaskan rentang tahun dan jumlah provinsi yang benar-benar dipakai di workflow deployment.


In [2]:
# Load clean dataset
df = pd.read_csv('../datasets/processed/clean_daya_beli.csv')
print(f"Dataset: {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Tahun: {df['Tahun'].min()}-{df['Tahun'].max()}, Provinsi: {df['Provinsi'].nunique()}")
print("Kolom:", df.columns.tolist())
df.head()


Dataset: 177 baris x 23 kolom
Tahun: 2021-2025, Provinsi: 38
Kolom: ['Provinsi', 'Tahun', 'Pengeluaran_Makanan', 'Pengeluaran_Bukan_Makanan', 'Total_Pengeluaran', 'UMP', 'TPT', 'TPAK', 'PDRB_HargaBerlaku', 'PDRB_HargaKonstan', 'Pct_Penduduk_Miskin', 'Inflasi_Rata_Tahunan', 'Gini_Rasio', 'IPM', 'Garis_Kemiskinan', 'Jumlah_Penduduk', 'Pct_Populasi', 'Pct_Akses_Air_Bersih', 'Protein_gram_per_hari', 'Inflasi_WB_Annual', 'GDP_PerCapita_PPP', 'Pct_Unemployment_WB', 'Poverty_Headcount_Pct']


,Provinsi,Tahun,Pengeluaran_Makanan,Pengeluaran_Bukan_Makanan,Total_Pengeluaran,UMP,TPT,TPAK,PDRB_HargaBerlaku,PDRB_HargaKonstan,...,IPM,Garis_Kemiskinan,Jumlah_Penduduk,Pct_Populasi,Pct_Akses_Air_Bersih,Protein_gram_per_hari,Inflasi_WB_Annual,GDP_PerCapita_PPP,Pct_Unemployment_WB,Poverty_Headcount_Pct
0,Aceh,2021,643591.40,494227.92,1137819.32,3165031.0,6.300,64.460,34673.56,25356.45,...,72.18,553270.25,5274.9,1.96,88.79,62.278819,1.560079,12757.074644,3.827,10.1
1,Bali,2021,628472.06,840152.05,1468624.11,2494000.0,5.395,73.625,50758.32,33123.79,...,75.69,447277.50,4317.4,1.60,97.56,62.278819,1.560079,12757.074644,3.827,10.1
2,Banten,2021,744893.26,766363.40,1511256.67,2460997.0,8.995,64.035,55383.29,38339.42,...,72.72,524712.25,11904.6,4.42,93.51,62.278819,1.560079,12757.074644,3.827,10.1
3,Bengkulu,2021,580273.31,558299.08,1138572.40,2215000.0,3.685,70.745,39167.13,23545.64,...,71.64,575415.25,2010.7,0.75,67.39,62.278819,1.560079,12757.074644,3.827,10.1
4,DI Yogyakarta,2021,594622.24,823248.11,1417870.35,1765000.0,4.420,73.165,40516.00,29115.86,...,80.22,469253.00,3668.7,1.36,95.69,62.278819,1.560079,12757.074644,3.827,10.1


In [3]:
# Quick EDA
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Target distribution
sns.histplot(df['Total_Pengeluaran'], kde=True, ax=axes[0,0])
axes[0,0].set_title('Distribusi Total Pengeluaran')

# Daya beli per provinsi (rata-rata)
prov_mean = df.groupby('Provinsi')['Total_Pengeluaran'].mean().sort_values(ascending=False)
prov_mean.plot(kind='barh', ax=axes[0,1])
axes[0,1].set_title('Rata-rata Daya Beli per Provinsi')

# Trend tahunan
trend = df.groupby('Tahun')['Total_Pengeluaran'].mean()
trend.plot(marker='o', ax=axes[0,2])
axes[0,2].set_title('Trend Daya Beli Nasional')
axes[0,2].set_ylabel('Rp')

# UMP vs Pengeluaran
sns.scatterplot(data=df, x='UMP', y='Total_Pengeluaran', hue='Tahun', ax=axes[1,0])
axes[1,0].set_title('UMP vs Daya Beli')

# Inflasi vs Pengeluaran
sns.scatterplot(data=df, x='Inflasi_Rata_Tahunan', y='Total_Pengeluaran', hue='Provinsi', 
                legend=False, ax=axes[1,1])
axes[1,1].set_title('Inflasi vs Daya Beli')

# PDRB vs Pengeluaran
sns.scatterplot(data=df, x='PDRB_HargaKonstan', y='Total_Pengeluaran', hue='Tahun', ax=axes[1,2])
axes[1,2].set_title('PDRB vs Daya Beli')

plt.tight_layout()
plt.show()

print("Insight:")
print(f"   - Daya beli tertinggi: {prov_mean.index[0]} (Rp {prov_mean.iloc[0]:,.0f})")
print(f"   - Daya beli terendah: {prov_mean.index[-1]} (Rp {prov_mean.iloc[-1]:,.0f})")
print(f"   - Rasio tertinggi/terendah: {prov_mean.iloc[0]/prov_mean.iloc[-1]:.1f}x")


Insight:
   - Daya beli tertinggi: DKI Jakarta (Rp 2,682,078)
   - Daya beli terendah: Nusa Tenggara Timur (Rp 938,112)
   - Rasio tertinggi/terendah: 2.9x


## 2. Feature Engineering

Fitur turunan dibuat untuk membantu model membaca sinyal ekonomi yang lebih dekat ke daya beli riil.

### Fitur utama
- `Real_UMP`: pendekatan sederhana untuk upah minimum riil
- growth per provinsi: perubahan tahunan pada UMP riil, PDRB, dan TPT
- interaction terms: hubungan gabungan antara upah, output, inflasi, dan pengangguran
- log transform: meredam skala ekstrem pada variabel ekonomi tertentu


In [4]:
print("Feature Engineering...")

# 1. Real UMP (purchasing power)
df['Real_UMP'] = df['UMP'] / (1 + df['Inflasi_Rata_Tahunan'])
print("   Real_UMP = UMP / (1 + Inflasi)")

# 2. Sort for time-series operations
df = df.sort_values(['Provinsi', 'Tahun']).reset_index(drop=True)

# 3. YoY Growth rates (within province)
growth_features = ['Total_Pengeluaran', 'Real_UMP', 'PDRB_HargaKonstan', 'TPT']
for col in growth_features:
    df[f'{col}_Growth'] = df.groupby('Provinsi')[col].pct_change() * 100
    print(f"   {col}_Growth (YoY %)")

# 4. Interaction terms (economic hypotheses)
df['UMP_x_PDRB'] = df['Real_UMP'] * df['PDRB_HargaKonstan']
df['Inflasi_x_TPT'] = df['Inflasi_Rata_Tahunan'] * df['TPT']
print("   Interaction: UMP x PDRB, Inflasi x TPT")

# 5. Log transforms for skewed variables
df['Log_Total_Pengeluaran'] = np.log1p(df['Total_Pengeluaran'])
df['Log_PDRB'] = np.log1p(df['PDRB_HargaKonstan'])
df['Log_UMP'] = np.log1p(df['Real_UMP'])
print("   Log transforms")

# 6. Impute missing values in provincial/WB features
wb_cols = [c for c in df.columns if 'WB' in c or 'GDP' in c or 'Poverty' in c or 'Unemployment' in c]
prov_cols = ['IPM', 'Jumlah_Penduduk', 'Pct_Populasi']
for cols in [wb_cols, prov_cols]:
    if cols and df[cols].isna().sum().sum() > 0:
        before = df[cols].isna().sum().sum()
        df[cols] = df.groupby('Provinsi')[cols].transform(lambda x: x.ffill().bfill())
        after = df[cols].isna().sum().sum()
        print(f"   [IMPUTASI] {before-after} NaN diisi di {cols}")

# Drop irrelevant columns
drop_cols = ['UMP', 'PDRB_HargaBerlaku', 'TPAK', 'Pct_Penduduk_Miskin',
             'Pengeluaran_Makanan', 'Pengeluaran_Bukan_Makanan']
df_model = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

# Drop rows with NaN in critical columns
critical_cols = ['Total_Pengeluaran', 'Real_UMP', 'PDRB_HargaKonstan', 'TPT', 'Inflasi_Rata_Tahunan']
df_model = df_model.dropna(subset=critical_cols).copy()

print(f"Dataset siap: {df_model.shape[0]} baris x {df_model.shape[1]} kolom")
print(f"   NaN tersisa: {df_model.isna().sum().sum()}")

df_model.head()


Feature Engineering...
   Real_UMP = UMP / (1 + Inflasi)
   Total_Pengeluaran_Growth (YoY %)
   Real_UMP_Growth (YoY %)
   PDRB_HargaKonstan_Growth (YoY %)
   TPT_Growth (YoY %)
   Interaction: UMP x PDRB, Inflasi x TPT
   Log transforms
   [IMPUTASI] 0 NaN diisi di ['IPM', 'Jumlah_Penduduk', 'Pct_Populasi']
Dataset siap: 177 baris x 27 kolom
   NaN tersisa: 159


,Provinsi,Tahun,Total_Pengeluaran,TPT,PDRB_HargaKonstan,Inflasi_Rata_Tahunan,Gini_Rasio,IPM,Garis_Kemiskinan,Jumlah_Penduduk,...,Real_UMP,Total_Pengeluaran_Growth,Real_UMP_Growth,PDRB_HargaKonstan_Growth,TPT_Growth,UMP_x_PDRB,Inflasi_x_TPT,Log_Total_Pengeluaran,Log_PDRB,Log_UMP
0,Aceh,2021,1137819.32,6.300,25356.45,0.155000,0.3235,72.18,553270.25,5274.9,...,2.740287e+06,NaN,NaN,NaN,NaN,6.948394e+10,0.976500,13.944625,10.140828,14.823573
1,Aceh,2022,1180132.93,6.070,26061.53,0.450000,0.3010,72.80,605322.00,5334.9,...,2.183766e+06,3.718834,-20.308864,2.780673,-3.650794,5.691227e+10,2.731500,13.981138,10.168254,14.596562
2,Aceh,2023,1225976.00,5.890,26800.14,0.215000,0.2960,73.40,634889.50,5409.2,...,2.809602e+06,3.884568,28.658577,2.834101,-2.965404,7.529772e+10,1.266350,14.019249,10.196200,14.848554
3,Aceh,2024,1264733.00,5.655,27684.18,0.130000,0.2940,74.03,672333.25,5554.8,...,3.062542e+06,3.161318,9.002698,3.298639,-3.989813,8.478395e+10,0.735150,14.050372,10.228653,14.934756
4,Aceh,2025,1299756.00,5.570,28144.60,0.243333,0.2780,74.03,705310.75,5554.8,...,2.964302e+06,2.769201,-3.207766,1.663116,-1.503095,8.342911e+10,1.355367,14.077688,10.245146,14.902153


## 3. Train-Test Split Kronologis

Split dibuat berdasarkan waktu, bukan random, agar evaluasi lebih mendekati situasi prediksi nyata.

- train: 2021-2023
- test: 2024-2025


In [5]:
# Chronological split: Train <=2023, Test >=2024
train = df_model[df_model['Tahun'] <= 2023].copy()
test = df_model[df_model['Tahun'] >= 2024].copy()
train_panel = train.set_index(['Provinsi', 'Tahun'])
test_panel = test.set_index(['Provinsi', 'Tahun'])

print(f"Train: {len(train)} ({train['Tahun'].min()}-{train['Tahun'].max()})")
print(f"Test:  {len(test)} ({test['Tahun'].min()}-{test['Tahun'].max()})")

y_train = train['Total_Pengeluaran'].values
y_test = test['Total_Pengeluaran'].values


Train: 102 (2021-2023)
Test:  75 (2024-2025)


## 4. Baseline OLS

Baseline ini dipakai sebagai pembanding interpretatif paling sederhana. Tujuannya bukan mengejar akurasi terbaik, tetapi memberi titik acuan yang mudah dipahami.


In [6]:
print("="*60)
print("SKENARIO 1: BASELINE OLS")
print("="*60)

features_s1 = ['Tahun', 'Inflasi_Rata_Tahunan', 'Real_UMP']
X_train_s1 = sm.add_constant(train[features_s1])
X_test_s1 = sm.add_constant(test[features_s1])

model_s1 = sm.OLS(y_train, X_train_s1).fit()
print(model_s1.summary())

# VIF
vif_data = pd.DataFrame()
vif_data['Feature'] = features_s1
vif_data['VIF'] = [variance_inflation_factor(X_train_s1[features_s1].values, i) for i in range(len(features_s1))]
print("VIF:", vif_data.to_string(index=False))

# Metrics
pred_s1_train = model_s1.predict(X_train_s1)
pred_s1_test = model_s1.predict(X_test_s1)
print_metrics(y_train, pred_s1_train, "Train")
print_metrics(y_test, pred_s1_test, "Test")


SKENARIO 1: BASELINE OLS
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.351
Model:                            OLS   Adj. R-squared:                  0.332
Method:                 Least Squares   F-statistic:                     17.70
Date:                Mon, 15 Jun 2026   Prob (F-statistic):           2.94e-09
Time:                        01:46:11   Log-Likelihood:                -1414.6
No. Observations:                 102   AIC:                             2837.
Df Residuals:                      98   BIC:                             2848.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const  

{'R2': 0.1827552493491743,
 'MAE': 256949.42178302174,
 'RMSE': np.float64(311074.26742630213)}

## 5. Fixed Effects PanelOLS Macro

Ini adalah model ekonometrika utama untuk membaca arah pengaruh variabel makro antar-provinsi tanpa harus membanjiri model dengan dummy provinsi.


In [7]:
print("="*60)
print("SKENARIO 2: FIXED EFFECTS PANELOLS (MODEL UTAMA)")
print("="*60)

fe_features = ['Inflasi_Rata_Tahunan', 'Real_UMP', 'PDRB_HargaKonstan', 'TPT']

# linearmodels PanelOLS (for beautiful econometric summary)
model_fe = PanelOLS(train_panel['Total_Pengeluaran'], train_panel[fe_features], entity_effects=True)
res_fe = model_fe.fit(cov_type='clustered', cluster_entity=True)
print(res_fe)

# Manual Fixed Effects (for sklearn-compatible prediction)
entity_means = {}
for col in ['Total_Pengeluaran'] + fe_features:
    entity_means[col] = train.groupby('Provinsi')[col].mean().to_dict()

train_demeaned = train.copy()
for col in ['Total_Pengeluaran'] + fe_features:
    train_demeaned[col] = train.apply(lambda r: r[col] - entity_means[col].get(r['Provinsi'], 0), axis=1)

model_fe_manual = sm.OLS(train_demeaned['Total_Pengeluaran'], train_demeaned[fe_features]).fit()

# Predict on test (add back entity means from training)
test_demeaned = test.copy()
for col in fe_features:
    test_demeaned[col] = test.apply(lambda r: r[col] - entity_means[col].get(r['Provinsi'], 0), axis=1)

pred_fe_test_dem = model_fe_manual.predict(test_demeaned[fe_features])
pred_fe_test = pred_fe_test_dem + test['Provinsi'].apply(lambda p: entity_means['Total_Pengeluaran'].get(p, 0))

pred_fe_train_dem = model_fe_manual.predict(train_demeaned[fe_features])
pred_fe_train = pred_fe_train_dem + train['Provinsi'].apply(lambda p: entity_means['Total_Pengeluaran'].get(p, 0))

print_metrics(y_train, pred_fe_train, "Train")
print_metrics(y_test, pred_fe_test, "Test")

# FE Feature Importance (t-statistics)
fe_importance = pd.DataFrame({
    'Feature': fe_features,
    'Coefficient': model_fe_manual.params.values,
    't_stat': np.abs(model_fe_manual.tvalues.values),
    'p_value': model_fe_manual.pvalues.values
}).sort_values('t_stat', ascending=False)

print("FE Feature Importance (|t-stat|):")
for _, row in fe_importance.iterrows():
    sig = "***" if row['p_value'] < 0.01 else "**" if row['p_value'] < 0.05 else "*" if row['p_value'] < 0.1 else ""
    print(f"   {row['Feature']}: coef={row['Coefficient']:+.2f}, |t|={row['t_stat']:.2f}, p={row['p_value']:.3f} {sig}")


SKENARIO 2: FIXED EFFECTS PANELOLS (MODEL UTAMA)
                          PanelOLS Estimation Summary                           
Dep. Variable:      Total_Pengeluaran   R-squared:                        0.7823
Estimator:                   PanelOLS   R-squared (Between):              0.6160
No. Observations:                 102   R-squared (Within):               0.7823
Date:                Mon, Jun 15 2026   R-squared (Overall):              0.6165
Time:                        01:46:11   Log-likelihood                   -1217.5
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      57.493
Entities:                          34   P-value                           0.0000
Avg Obs:                       3.0000   Distribution:                    F(4,64)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-statistic (robust)

## 6. Fixed Effects PanelOLS Full

Versi full dipakai untuk menguji apakah penambahan banyak fitur benar-benar membantu. Bagian ini berguna sebagai stress test terhadap overfitting dan stabilitas koefisien.


In [8]:
print("="*60)
print("SKENARIO 3: FULL MODEL PANELOLS")
print("="*60)

# Exclude target-derived features to prevent data leakage
exclude_derived = ['Total_Pengeluaran_Growth', 'Log_Total_Pengeluaran']
all_numeric = [c for c in df_model.columns 
               if c not in ['Provinsi', 'Tahun', 'Total_Pengeluaran', 'Log_Total_Pengeluaran'] + exclude_derived
               and df_model[c].dtype in ['float64', 'int64']]
all_numeric = [c for c in all_numeric if df_model[c].isna().sum() < 50]
print(f"Features ({len(all_numeric)}): {all_numeric}")

# Drop rows with NaN
train_full = train.dropna(subset=['Total_Pengeluaran'] + all_numeric).copy()
test_full = test.dropna(subset=['Total_Pengeluaran'] + all_numeric).copy()
print(f"Rows after NaN drop - Train: {len(train_full)}, Test: {len(test_full)}")

y_train_full = train_full['Total_Pengeluaran'].values
y_test_full = test_full['Total_Pengeluaran'].values

entity_means_full = {}
for col in ['Total_Pengeluaran'] + all_numeric:
    entity_means_full[col] = train_full.groupby('Provinsi')[col].mean().to_dict()

train_dem_full = train_full.copy()
for col in ['Total_Pengeluaran'] + all_numeric:
    train_dem_full[col] = train_full.apply(lambda r: r[col] - entity_means_full[col].get(r['Provinsi'], 0), axis=1)

model_fe_full_manual = sm.OLS(train_dem_full['Total_Pengeluaran'], train_dem_full[all_numeric]).fit()

test_dem_full = test_full.copy()
for col in all_numeric:
    test_dem_full[col] = test_full.apply(lambda r: r[col] - entity_means_full[col].get(r['Provinsi'], 0), axis=1)

pred_s3_test_dem = model_fe_full_manual.predict(test_dem_full[all_numeric])
pred_s3_test = pred_s3_test_dem + test_full['Provinsi'].apply(lambda p: entity_means_full['Total_Pengeluaran'].get(p, 0))
pred_s3_train_dem = model_fe_full_manual.predict(train_dem_full[all_numeric])
pred_s3_train = pred_s3_train_dem + train_full['Provinsi'].apply(lambda p: entity_means_full['Total_Pengeluaran'].get(p, 0))

print_metrics(y_train_full, pred_s3_train, "Train")
print_metrics(y_test_full, pred_s3_test, "Test")

fe_full_imp = pd.DataFrame({
    'Feature': all_numeric,
    'Coef': model_fe_full_manual.params.values,
    't_stat': np.abs(model_fe_full_manual.tvalues.values),
    'p_value': model_fe_full_manual.pvalues.values
}).sort_values('t_stat', ascending=False)
print("Top 10 FE Full:")
print(fe_full_imp.head(10)[['Feature', 'Coef', 't_stat', 'p_value']].to_string(index=False))


SKENARIO 3: FULL MODEL PANELOLS
Features (22): ['TPT', 'PDRB_HargaKonstan', 'Inflasi_Rata_Tahunan', 'Gini_Rasio', 'IPM', 'Garis_Kemiskinan', 'Jumlah_Penduduk', 'Pct_Populasi', 'Pct_Akses_Air_Bersih', 'Protein_gram_per_hari', 'Inflasi_WB_Annual', 'GDP_PerCapita_PPP', 'Pct_Unemployment_WB', 'Poverty_Headcount_Pct', 'Real_UMP', 'Real_UMP_Growth', 'PDRB_HargaKonstan_Growth', 'TPT_Growth', 'UMP_x_PDRB', 'Inflasi_x_TPT', 'Log_PDRB', 'Log_UMP']
Rows after NaN drop - Train: 68, Test: 68
Train R2: 0.9988 | MAE: Rp 9,207 | RMSE: Rp 11,387
Test R2: -23.6300 | MAE: Rp 1,766,889 | RMSE: Rp 1,779,067
Top 10 FE Full:
              Feature          Coef   t_stat  p_value
                  TPT -1.250269e+05 4.888564 0.000011
Protein_gram_per_hari  3.539994e-01 4.731526 0.000018
    GDP_PerCapita_PPP  1.713313e+03 4.731526 0.000018
  Pct_Unemployment_WB -4.747989e-01 4.731526 0.000018
    Inflasi_WB_Annual -1.665120e+00 4.731526 0.000018
Poverty_Headcount_Pct -3.083110e-01 4.731526 0.000018
 Inflasi_Rat

## 7. Lasso Feature Selection

Lasso membantu melihat apakah ada fitur yang secara otomatis bisa ditekan menuju nol, sehingga model menjadi lebih ringkas dan mudah dijelaskan.


In [9]:
print("="*60)
print("SKENARIO 4: LASSO FEATURE SELECTION")
print("="*60)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(train_full[all_numeric])
X_test_scaled = scaler.transform(test_full[all_numeric])

alphas = np.logspace(-3, 2, 30)
lasso_cv = GridSearchCV(Lasso(max_iter=20000, random_state=42), {'alpha': alphas}, 
                        cv=5, scoring='neg_mean_squared_error')
lasso_cv.fit(X_train_scaled, y_train_full)
print(f"Best alpha: {lasso_cv.best_params_['alpha']:.4f}")

lasso = Lasso(alpha=lasso_cv.best_params_['alpha'], max_iter=20000, random_state=42)
lasso.fit(X_train_scaled, y_train_full)

pred_lasso_train = lasso.predict(X_train_scaled)
pred_lasso_test = lasso.predict(X_test_scaled)
print_metrics(y_train_full, pred_lasso_train, "Train")
print_metrics(y_test_full, pred_lasso_test, "Test")

lasso_imp = pd.DataFrame({'Feature': all_numeric, 'Coefficient': lasso.coef_})
lasso_selected = lasso_imp[lasso_imp['Coefficient'] != 0].copy()
lasso_selected['abs_coef'] = np.abs(lasso_selected['Coefficient'])
lasso_selected = lasso_selected.sort_values('abs_coef', ascending=False)

print(f"Selected: {len(lasso_selected)}/{len(all_numeric)} features")
print(lasso_selected[['Feature', 'Coefficient']].to_string(index=False))


SKENARIO 4: LASSO FEATURE SELECTION


Best alpha: 45.2035
Train R2: 0.9356 | MAE: Rp 71,292 | RMSE: Rp 82,638
Test R2: 0.3364 | MAE: Rp 253,251 | RMSE: Rp 292,025
Selected: 22/22 features
                 Feature   Coefficient
              UMP_x_PDRB  3.576814e+05
                Real_UMP -2.851530e+05
       PDRB_HargaKonstan -2.832665e+05
    Inflasi_Rata_Tahunan -2.758575e+05
                 Log_UMP  2.162469e+05
         Real_UMP_Growth -2.114506e+05
        Garis_Kemiskinan  1.577833e+05
            Pct_Populasi -1.494608e+05
         Jumlah_Penduduk  1.399031e+05
                Log_PDRB  9.692530e+04
              Gini_Rasio  8.795819e+04
                     IPM  8.260132e+04
           Inflasi_x_TPT  4.799065e+04
              TPT_Growth -3.809428e+04
                     TPT  1.705195e+04
    Pct_Akses_Air_Bersih -1.137514e+04
PDRB_HargaKonstan_Growth -5.942509e+03
   Protein_gram_per_hari  6.930413e+00
   Poverty_Headcount_Pct -1.032544e-11
     Pct_Unemployment_WB -5.189470e-12
       GDP_PerCapita_PPP  4.761

## 8. Ridge Deployment Path

Bagian ini sengaja mengikuti helper resmi yang sama dengan aplikasi web.

### Kenapa dipisahkan dari eksperimen lain
- notebook analisis boleh kaya metode, tetapi artefak web harus punya satu sumber kebenaran
- fitur deployment dijaga tetap leakage-free
- metrik yang keluar dari sini adalah metrik yang seharusnya kita pakai saat membicarakan performa website


In [10]:
from pathlib import Path
import sys

print('='*60)
print('SKENARIO 5: RETRAIN RIDGE DEPLOYMENT (JALUR WEB RESMI)')
print('='*60)

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
dashboard_dir = project_root / 'dashboard'
if str(dashboard_dir) not in sys.path:
    sys.path.insert(0, str(dashboard_dir))

from train_daya_beli_ridge import main as retrain_ridge_deployment

ridge_bundle = retrain_ridge_deployment()
print('Artifact deployment tersimpan di ../models/best_daya_beli_ridge.pkl')
print(f"Test R2 deployment: {ridge_bundle['test_r2']:.4f}")
print(f"Test MAE deployment: Rp {ridge_bundle['test_mae']:,.0f}")
print(f"Walk-forward mean R2: {ridge_bundle['walk_forward']['mean']['r2']:.4f}")


SKENARIO 5: RETRAIN RIDGE DEPLOYMENT (JALUR WEB RESMI)


Saved: C:\Users\adief\OneDrive\Dokumen\Semester 6\Machine Learning\Week 1-12 (Project Testing)\models\best_daya_beli_ridge.pkl
Rows: 177 | Provinces: 38
Years: 2021 - 2025
Features: 18 numeric + 1 categorical
Excluded target components: Pengeluaran_Makanan, Pengeluaran_Bukan_Makanan
Best alpha: 1.0
Train R2: 0.9883 | Test R2: 0.8744
Test MAE: 104371.62 | Test RMSE: 121936.4
Walk-forward mean: {'r2': 0.9306, 'mae': 65705.35, 'rmse': 85580.56}
Artifact deployment tersimpan di ../models/best_daya_beli_ridge.pkl
Test R2 deployment: 0.8744
Test MAE deployment: Rp 104,372
Walk-forward mean R2: 0.9306


## 9. Random Forest dan XGBoost

Model non-linear ini dipakai sebagai pembanding. Mereka berguna untuk melihat apakah ada pola kompleks yang tidak tertangkap model linear atau panel klasik.


In [11]:
print("="*60)
print("SKENARIO 5: RANDOM FOREST & XGBOOST")
print("="*60)

# Random Forest
rf = RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_split=5,
                             min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(train_full[all_numeric], y_train_full)
pred_rf_train = rf.predict(train_full[all_numeric])
pred_rf_test = rf.predict(test_full[all_numeric])
print_metrics(y_train_full, pred_rf_train, "RF Train")
print_metrics(y_test_full, pred_rf_test, "RF Test")

# XGBoost
xgb = XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb.fit(train_full[all_numeric], y_train_full, eval_set=[(test_full[all_numeric], y_test_full)], verbose=False)
pred_xgb_train = xgb.predict(train_full[all_numeric])
pred_xgb_test = xgb.predict(test_full[all_numeric])
print_metrics(y_train_full, pred_xgb_train, "XGB Train")
print_metrics(y_test_full, pred_xgb_test, "XGB Test")

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_imp = pd.DataFrame({'Feature': all_numeric, 'Importance': rf.feature_importances_})
rf_imp = rf_imp.sort_values('Importance', ascending=True).tail(10)
rf_imp.plot(x='Feature', y='Importance', kind='barh', ax=axes[0], legend=False)
axes[0].set_title('Random Forest - Top 10 Feature Importance')

xgb_imp = pd.DataFrame({'Feature': all_numeric, 'Importance': xgb.feature_importances_})
xgb_imp = xgb_imp.sort_values('Importance', ascending=True).tail(10)
xgb_imp.plot(x='Feature', y='Importance', kind='barh', ax=axes[1], legend=False, color='orange')
axes[1].set_title('XGBoost - Top 10 Feature Importance')

plt.tight_layout()
plt.show()


SKENARIO 5: RANDOM FOREST & XGBOOST


RF Train R2: 0.8856 | MAE: Rp 68,141 | RMSE: Rp 110,148
RF Test R2: 0.7224 | MAE: Rp 136,026 | RMSE: Rp 188,879


XGB Train R2: 1.0000 | MAE: Rp 58 | RMSE: Rp 75
XGB Test R2: 0.8611 | MAE: Rp 108,566 | RMSE: Rp 133,579


## 10. Ringkasan Perbandingan Model

Di sini semua model diletakkan dalam satu tabel agar trade-off akurasi, stabilitas, dan risiko overfit lebih mudah dibaca.


In [12]:
print('='*60)
print('PERBANDINGAN SEMUA MODEL')
print('='*60)

m_s1_train = print_metrics(y_train, pred_s1_train, 'Train')
m_s1_test = print_metrics(y_test, pred_s1_test, 'Test')
m_s2_train = print_metrics(y_train, pred_fe_train, 'Train')
m_s2_test = print_metrics(y_test, pred_fe_test, 'Test')
m_s3_train = print_metrics(y_train_full, pred_s3_train, 'Train')
m_s3_test = print_metrics(y_test_full, pred_s3_test, 'Test')
m_s4_train = print_metrics(y_train_full, pred_lasso_train, 'Train')
m_s4_test = print_metrics(y_test_full, pred_lasso_test, 'Test')
m_rf_train = print_metrics(y_train_full, pred_rf_train, 'Train')
m_rf_test = print_metrics(y_test_full, pred_rf_test, 'Test')
m_xgb_train = print_metrics(y_train_full, pred_xgb_train, 'Train')
m_xgb_test = print_metrics(y_test_full, pred_xgb_test, 'Test')
m_ridge_train = {'R2': ridge_bundle['train_r2'], 'MAE': ridge_bundle['train_mae'], 'RMSE': ridge_bundle['train_rmse']}
m_ridge_test = {'R2': ridge_bundle['test_r2'], 'MAE': ridge_bundle['test_mae'], 'RMSE': ridge_bundle['test_rmse']}
print(f"Ridge deployment -> Train R2: {m_ridge_train['R2']:.4f} | Test R2: {m_ridge_test['R2']:.4f}")

comparison = pd.DataFrame({
    'Model': ['OLS Baseline', 'Panel FE Macro', 'Panel FE Full', 'Lasso', 'Ridge Deployment', 'Random Forest', 'XGBoost'],
    'Train_R2': [m_s1_train['R2'], m_s2_train['R2'], m_s3_train['R2'], m_s4_train['R2'], m_ridge_train['R2'], m_rf_train['R2'], m_xgb_train['R2']],
    'Test_R2': [m_s1_test['R2'], m_s2_test['R2'], m_s3_test['R2'], m_s4_test['R2'], m_ridge_test['R2'], m_rf_test['R2'], m_xgb_test['R2']],
    'Test_MAE': [m_s1_test['MAE'], m_s2_test['MAE'], m_s3_test['MAE'], m_s4_test['MAE'], m_ridge_test['MAE'], m_rf_test['MAE'], m_xgb_test['MAE']],
    'Test_RMSE': [m_s1_test['RMSE'], m_s2_test['RMSE'], m_s3_test['RMSE'], m_s4_test['RMSE'], m_ridge_test['RMSE'], m_rf_test['RMSE'], m_xgb_test['RMSE']]
})
comparison['Overfit_Gap'] = comparison['Train_R2'] - comparison['Test_R2']
comparison = comparison.sort_values('Test_R2', ascending=False)

print('Perbandingan Model (diurutkan Test R2):')
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
x = np.arange(len(comparison))
width = 0.35
axes[0].bar(x - width/2, comparison['Train_R2'], width, label='Train R2', alpha=0.8)
axes[0].bar(x + width/2, comparison['Test_R2'], width, label='Test R2', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison['Model'], rotation=30, ha='right')
axes[0].set_ylabel('R2')
axes[0].set_title('Train vs Test R2')
axes[0].legend()
axes[0].set_ylim(0, 1.1)

axes[1].bar(comparison['Model'], comparison['Overfit_Gap'], color='coral')
axes[1].set_ylabel('Train R2 - Test R2')
axes[1].set_title('Overfitting Gap (lebih kecil = lebih baik)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

best = comparison.iloc[0]
print(f"BEST MODEL: {best['Model']} | Test R2: {best['Test_R2']:.4f} | Test MAE: Rp {best['Test_MAE']:,.0f}")



PERBANDINGAN SEMUA MODEL
Train R2: 0.3514 | MAE: Rp 200,854 | RMSE: Rp 255,295
Test R2: 0.1828 | MAE: Rp 256,949 | RMSE: Rp 311,074
Train R2: 0.9864 | MAE: Rp 28,229 | RMSE: Rp 36,961
Test R2: 0.4273 | MAE: Rp 126,334 | RMSE: Rp 260,413
Train R2: 0.9988 | MAE: Rp 9,207 | RMSE: Rp 11,387
Test R2: -23.6300 | MAE: Rp 1,766,889 | RMSE: Rp 1,779,067
Train R2: 0.9356 | MAE: Rp 71,292 | RMSE: Rp 82,638
Test R2: 0.3364 | MAE: Rp 253,251 | RMSE: Rp 292,025
Train R2: 0.8856 | MAE: Rp 68,141 | RMSE: Rp 110,148
Test R2: 0.7224 | MAE: Rp 136,026 | RMSE: Rp 188,879
Train R2: 1.0000 | MAE: Rp 58 | RMSE: Rp 75
Test R2: 0.8611 | MAE: Rp 108,566 | RMSE: Rp 133,579
Ridge deployment -> Train R2: 0.9883 | Test R2: 0.8744
Perbandingan Model (diurutkan Test R2):
           Model  Train_R2    Test_R2     Test_MAE    Test_RMSE  Overfit_Gap
Ridge Deployment  0.988298   0.874429 1.043716e+05 1.219364e+05     0.113870
         XGBoost  1.000000   0.861147 1.085665e+05 1.335789e+05     0.138853
   Random Forest  

## 11. Feature Importance Ekonomi

Tujuannya bukan sekadar melihat angka besar-kecil, tetapi memeriksa apakah model tetap menaruh perhatian pada variabel ekonomi yang masuk akal secara substantif.


In [13]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# PanelOLS Macro
top_fe = fe_importance.head(10)
axes[0,0].barh(top_fe['Feature'], top_fe['t_stat'], color='steelblue')
axes[0,0].set_xlabel('|t-statistic|')
axes[0,0].set_title('S2: Panel FE (Macro) - |t-stat|')
axes[0,0].invert_yaxis()

# PanelOLS Full
top_fe_full = fe_full_imp.head(10)
axes[0,1].barh(top_fe_full['Feature'], top_fe_full['t_stat'], color='forestgreen')
axes[0,1].set_xlabel('|t-statistic|')
axes[0,1].set_title('S3: Panel FE (Full) - |t-stat|')
axes[0,1].invert_yaxis()

# Lasso coefficients
axes[1,0].barh(lasso_selected['Feature'], lasso_selected['Coefficient'], color='coral')
axes[1,0].set_xlabel('Coefficient')
axes[1,0].set_title('S4: Lasso - Selected Features')
axes[1,0].invert_yaxis()

# XGBoost
axes[1,1].barh(xgb_imp['Feature'], xgb_imp['Importance'], color='orange')
axes[1,1].set_xlabel('Feature Importance (gain)')
axes[1,1].set_title('S5: XGBoost - Feature Importance')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.show()

print("Catatan: Semua feature di atas adalah VARIABEL EKONOMI MAKRO, bukan dummy provinsi!")


Catatan: Semua feature di atas adalah VARIABEL EKONOMI MAKRO, bukan dummy provinsi!


## 12. Elastisitas Daya Beli

Estimasi ini membantu menerjemahkan model menjadi bahasa kebijakan: jika suatu variabel bergerak 1 persen, kira-kira seberapa sensitif daya beli terhadap perubahan itu.


In [14]:
print("="*60)
print("ELASTISITAS DAYA BELI (dari Panel FE Macro)")
print("="*60)

base_pred = pred_fe_test.copy()
elasticities = {}
for feat in fe_features:
    test_perturbed = test.copy()
    test_perturbed[feat] = test_perturbed[feat] * 1.01
    test_dem_pert = test_perturbed.copy()
    for col in fe_features:
        test_dem_pert[col] = test_perturbed.apply(lambda r: r[col] - entity_means[col].get(r['Provinsi'], 0), axis=1)
    pred_pert = model_fe_manual.predict(test_dem_pert[fe_features])
    pred_pert = pred_pert + test['Provinsi'].apply(lambda p: entity_means['Total_Pengeluaran'].get(p, 0))
    elasticity = np.mean((pred_pert - base_pred) / base_pred) * 100
    elasticities[feat] = elasticity

elasticity_df = pd.DataFrame(list(elasticities.items()), columns=['Variable', 'Elasticity'])
elasticity_df = elasticity_df.sort_values('Elasticity', key=abs, ascending=False)
print(elasticity_df.to_string(index=False))

print("Interpretasi:")
for _, row in elasticity_df.iterrows():
    direction = "naik" if row['Elasticity'] > 0 else "turun"
    print(f"   Jika {row['Variable']} naik 1% -> daya beli {direction} {abs(row['Elasticity']):.2f}%")


ELASTISITAS DAYA BELI (dari Panel FE Macro)
            Variable  Elasticity
            Real_UMP    0.507236
                 TPT   -0.301445
   PDRB_HargaKonstan    0.256345
Inflasi_Rata_Tahunan    0.046483
Interpretasi:
   Jika Real_UMP naik 1% -> daya beli naik 0.51%
   Jika TPT naik 1% -> daya beli turun 0.30%
   Jika PDRB_HargaKonstan naik 1% -> daya beli naik 0.26%
   Jika Inflasi_Rata_Tahunan naik 1% -> daya beli naik 0.05%


## 13. Simulasi Counterfactual

Bagian ini mengubah model menjadi alat diskusi kebijakan. Skenario yang dipakai bukan ramalan pasti, melainkan eksperimen terarah untuk membaca arah dampak.


In [15]:
print("="*60)
print("SIMULASI COUNTERFACTUAL")
print("="*60)

scenarios = {
    'Inflasi +5%': lambda df: df.assign(Inflasi_Rata_Tahunan=df['Inflasi_Rata_Tahunan']*1.05),
    'UMP +10%': lambda df: df.assign(Real_UMP=df['Real_UMP']*1.10),
    'Resesi (TPT x3)': lambda df: df.assign(TPT=df['TPT']*3),
    'PDRB -20%': lambda df: df.assign(PDRB_HargaKonstan=df['PDRB_HargaKonstan']*0.80),
    'Inflasi+5% & UMP+10%': lambda df: df.assign(
        Inflasi_Rata_Tahunan=df['Inflasi_Rata_Tahunan']*1.05,
        Real_UMP=df['Real_UMP']*1.10)
}

results = []
for name, transform in scenarios.items():
    test_sc = transform(test.copy())
    test_dem_sc = test_sc.copy()
    for col in fe_features:
        test_dem_sc[col] = test_sc.apply(lambda r: r[col] - entity_means[col].get(r['Provinsi'], 0), axis=1)
    pred_sc = model_fe_manual.predict(test_dem_sc[fe_features])
    pred_sc = pred_sc + test['Provinsi'].apply(lambda p: entity_means['Total_Pengeluaran'].get(p, 0))
    change = (pred_sc.mean() - base_pred.mean()) / base_pred.mean() * 100
    results.append({'Skenario': name, 'Avg_Daya_Beli_Rp': pred_sc.mean(), 'Perubahan_%': change})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['green' if x > 0 else 'red' for x in results_df['Perubahan_%']]
bars = ax.barh(results_df['Skenario'], results_df['Perubahan_%'], color=colors, alpha=0.7)
ax.set_xlabel('Perubahan Daya Beli Rata-rata (%)')
ax.set_title('Dampak Skenario Counterfactual terhadap Daya Beli')
ax.axvline(0, color='black', linestyle='--')

for bar, val in zip(bars, results_df['Perubahan_%']):
    ax.text(val + 0.2 if val > 0 else val - 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', ha='left' if val > 0 else 'right', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("Ringkasan Policy Insight:")
for _, row in results_df.iterrows():
    direction = "naik" if row['Perubahan_%'] > 0 else "turun"
    print(f"   {row['Skenario']}: daya beli {direction} {abs(row['Perubahan_%']):.1f}%")


SIMULASI COUNTERFACTUAL
            Skenario  Avg_Daya_Beli_Rp  Perubahan_%
         Inflasi +5%      1.482735e+06     0.214596
            UMP +10%      1.548701e+06     4.673133
     Resesi (TPT x3)      6.505215e+05   -56.032759
           PDRB -20%      1.400825e+06    -5.321461
Inflasi+5% & UMP+10%      1.551876e+06     4.887729
Ringkasan Policy Insight:
   Inflasi +5%: daya beli naik 0.2%
   UMP +10%: daya beli naik 4.7%
   Resesi (TPT x3): daya beli turun 56.0%
   PDRB -20%: daya beli turun 5.3%
   Inflasi+5% & UMP+10%: daya beli naik 4.9%


## 14. Diagnostik Residual

Diagnostik ini dipakai untuk menilai apakah pola error masih menyimpan masalah penting seperti heteroskedastisitas, distribusi yang terlalu berat ekornya, atau bias sistematis.


In [16]:
print("="*60)
print("DIAGNOSTIK RESIDUAL (Panel FE Macro)")
print("="*60)

residuals = y_test - pred_fe_test

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Residuals vs Fitted
axes[0,0].scatter(pred_fe_test, residuals, alpha=0.6, edgecolors='k', linewidth=0.5)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Residuals')
axes[0,0].set_title('Residuals vs Fitted')

# 2. Histogram of residuals
sns.histplot(residuals, kde=True, ax=axes[0,1])
axes[0,1].set_title('Distribusi Residual')
axes[0,1].axvline(0, color='red', linestyle='--')

# 3. Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[0,2])
axes[0,2].set_title('Q-Q Plot (Normalitas)')

# 4. Actual vs Predicted
axes[1,0].scatter(y_test, pred_fe_test, alpha=0.6, edgecolors='k', linewidth=0.5)
min_val = min(y_test.min(), pred_fe_test.min())
max_val = max(y_test.max(), pred_fe_test.max())
axes[1,0].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')
axes[1,0].set_xlabel('Actual')
axes[1,0].set_ylabel('Predicted')
axes[1,0].set_title('Actual vs Predicted')
axes[1,0].legend()

# 5. Residuals by Province (boxplot)
resid_df = pd.DataFrame({'Provinsi': test['Provinsi'].values, 'Residual': residuals})
top_provs = resid_df['Provinsi'].value_counts().head(10).index
resid_top = resid_df[resid_df['Provinsi'].isin(top_provs)]
sns.boxplot(data=resid_top, x='Provinsi', y='Residual', ax=axes[1,1])
axes[1,1].set_xticklabels(axes[1,1].get_xticklabels(), rotation=45, ha='right')
axes[1,1].set_title('Residuals by Province (Top 10)')
axes[1,1].axhline(0, color='red', linestyle='--')

# 6. Residuals over time
resid_df['Tahun'] = test['Tahun'].values
sns.boxplot(data=resid_df, x='Tahun', y='Residual', ax=axes[1,2])
axes[1,2].set_title('Residuals over Time')
axes[1,2].axhline(0, color='red', linestyle='--')

plt.tight_layout()
plt.show()

# Tests
jb_stat, jb_p = stats.jarque_bera(residuals)
sw_stat, sw_p = stats.shapiro(residuals)
print(f"Jarque-Bera: statistic={jb_stat:.2f}, p-value={jb_p:.4f}")
print(f"Shapiro-Wilk: statistic={sw_stat:.4f}, p-value={sw_p:.4f}")
print(f"Normalitas: {'Normal' if jb_p > 0.05 else 'Tidak normal (tapi tolerable untuk n>30)'}")

bp_stat, bp_p, _, _ = het_breuschpagan(residuals, sm.add_constant(test_demeaned[fe_features]))
print(f"Breusch-Pagan: LM={bp_stat:.2f}, p-value={bp_p:.4f}")
print(f"Heteroskedastisitas: {'Homoskedastis' if bp_p > 0.05 else 'Heteroskedastis -> gunakan robust SE'}")


DIAGNOSTIK RESIDUAL (Panel FE Macro)


Jarque-Bera: statistic=356.85, p-value=0.0000
Shapiro-Wilk: statistic=0.6464, p-value=0.0000
Normalitas: Tidak normal (tapi tolerable untuk n>30)
Breusch-Pagan: LM=47.58, p-value=0.0000
Heteroskedastisitas: Heteroskedastis -> gunakan robust SE


## 15. Export Artefak Analisis

Notebook ini menulis ulang ringkasan analisis pendukung, sementara artefak Ridge deployment tetap diproduksi dari helper resmi agar konsisten dengan aplikasi web.


In [17]:
print('='*60)
print('EXPORT RINGKASAN ANALISIS')
print('='*60)

os.makedirs('../models', exist_ok=True)
comparison.to_csv('../models/model_comparison.csv', index=False)
print('Comparison saved: ../models/model_comparison.csv')

xgb_bundle = {'model': xgb, 'features': all_numeric, 'type': 'XGBoost'}
joblib.dump(xgb_bundle, '../models/best_daya_beli_xgboost.pkl')
print('Saved: ../models/best_daya_beli_xgboost.pkl')
print('Artifact Ridge deployment tetap mengikuti helper train_daya_beli_ridge.py agar konsisten dengan website.')
print('DONE!')


EXPORT RINGKASAN ANALISIS
Comparison saved: ../models/model_comparison.csv
Saved: ../models/best_daya_beli_xgboost.pkl
Artifact Ridge deployment tetap mengikuti helper train_daya_beli_ridge.py agar konsisten dengan website.
DONE!


## Kesimpulan

### Ringkasan utama
1. Jalur deployment Ridge sekarang sudah disatukan dengan workflow website.
2. Scope 2021-2025 lebih defensible dibanding menarik tahun yang fitur pendukungnya belum lengkap.
3. Model non-linear seperti XGBoost tetap kuat, tetapi Ridge deployment masih menang dari sisi stabilitas dan konsistensi integrasi web.
4. Panel FE tetap paling berguna untuk interpretasi kebijakan, walaupun bukan artefak deployment utama.

### Cara membaca notebook ini
- gunakan bagian Panel FE saat ingin menjelaskan arah pengaruh ekonomi
- gunakan bagian Ridge deployment saat ingin membicarakan performa model yang benar-benar hidup di aplikasi
- gunakan counterfactual sebagai alat ilustrasi kebijakan, bukan kepastian angka masa depan
